In [ ]:
import pandas as pd

In [ ]:
df_train = pd.read_csv('../dataset/train.csv')
df_test = pd.read_csv('../dataset/test.csv')
df_original = pd.read_csv('../dataset/original.csv')

In [ ]:
df_train = df_train.drop(['id', 'CustomerId'], axis=1)
df_original = df_original.drop(['RowNumber', 'CustomerId'], axis=1)

In [ ]:
df_original.head()

In [ ]:
df_train.head()

In [ ]:
df_train = pd.concat([df_train, df_original], axis=0)

In [ ]:
df_train = df_train.drop_duplicates()
df_train = df_train.dropna()

In [ ]:
df_train.isna().sum()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import gc

def dummy(text):
    return text

vectorizer = TfidfVectorizer(ngram_range=(3, 5), 
                             lowercase=False, 
                             sublinear_tf=True, 
                             analyzer='word',
                             tokenizer=dummy,
                             preprocessor=dummy,
                             token_pattern=None, 
                             strip_accents='unicode', 
                             max_features=5000)

train_surnames = vectorizer.fit_transform(df_train["Surname"])

test_surnames = vectorizer.transform(df_test["Surname"])

del vectorizer
gc.collect()

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=100)
tfidf_train_pca = pca.fit_transform(train_surnames.toarray())
tfidf_test_pca = pca.transform(test_surnames.toarray())

pca_columns = [f'Surname_PCA_{i+1}' for i in range(100)]
df_train_pca = pd.DataFrame(tfidf_train_pca, columns=pca_columns)
df_test_pca = pd.DataFrame(tfidf_test_pca, columns=pca_columns)

In [ ]:
df_train_pca

In [ ]:
df_train.reset_index(drop=True, inplace=True)
df_train_pca.reset_index(drop=True, inplace=True)

df_train = pd.concat([df_train, df_train_pca], axis="columns")

In [ ]:
df_test.reset_index(drop=True, inplace=True)
df_test_pca.reset_index(drop=True, inplace=True)

df_test = pd.concat([df_test, df_test_pca], axis="columns")

In [ ]:
df_train = df_train.drop("Surname", axis=1)
df_test  = df_test.drop("Surname", axis=1)

In [ ]:
df_train.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

columns = ['Gender'] 

for col in df_train[columns]:
    enc = label_encoder.fit(df_train[col])
    encoded = label_encoder.transform(df_train[col])
    df_train[col] = encoded
    
for col in df_test[columns]:
    enc = label_encoder.fit(df_test[col])
    encoded = label_encoder.transform(df_test[col])
    df_test[col] = encoded

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output = False).set_output(transform='pandas')

ohetransform = ohe.fit_transform(df_train[['Geography']])
ohetransform

df_train = pd.concat([df_train, ohetransform], axis=1).drop(columns=["Geography"])

ohetransform = ohe.fit_transform(df_test[['Geography']])
ohetransform

df_test = pd.concat([df_test, ohetransform], axis=1).drop(columns=["Geography"])

In [ ]:
X = df_train.drop('Exited', axis=1)
y = df_train['Exited']


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("Training set class distribution:\n", y_train.value_counts())
print("Testing set class distribution:\n", y_test.value_counts())

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

param_grid = {
    'n_estimators': [100],
    'max_depth': [5],
    'learning_rate': [0.1],
    'min_samples_split': [2],
    'min_samples_leaf': [8]
}

grid_search = GridSearchCV(estimator=GradientBoostingRegressor(random_state=0),
                           param_grid=param_grid,
                           cv=2,
                           scoring='neg_mean_squared_error',
                           n_jobs=-1,
                           verbose=2)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)

best_gbr = grid_search.best_estimator_

y_pred_best = best_gbr.predict(X_test)

mse_best = mean_squared_error(y_test, y_pred_best)
print("Best Mean Squared Error:", mse_best)
